# Activity recognition with a hidden Markov model

Formative 2, Machine Learning Techniques. Four hidden states — standing, walking, jumping, and still — are decoded from windowed accelerometer and gyroscope features.

The `PLACEHOLDER` CSVs are empty templates, and the manifest marks them `placeholder` so training skips them. Replace each file with a real export and set `source` to `sensor-logger`. Video links for the four activities are listed in the README as `PASTE_LINK`.

The model is a diagonal Gaussian HMM in NumPy: Baum-Welch for learning, Viterbi for the label sequence. Nothing is imported from an HMM library.


In [ ]:
from pathlib import Path
import sys

import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluate import confusion_matrix, per_class_metrics
from src.experiment import fit_model, load_splits
from src.features import ACTIVITIES, FEATURE_NAMES, HOP, TARGET_FS, WINDOW
from src.hmm import self_test
from src.plots import plot_decoded, plot_heatmap, plot_likelihood, plot_raw_magnitude

check = self_test()
print(
    f"Implementation check passed. "
    f"Viterbi accuracy on a known 2-state chain: {check['viterbi_accuracy']:.2f}. "
    f"EM log-likelihood {check['loglik_start']:.2f} -> {check['loglik_end']:.2f} "
    f"in {check['iterations']} iterations, and it did not fall."
)


## Data

Each file has `timestamp` (seconds), accelerometer and gyroscope axes, and an `activity` label. Single-activity files last 6.5–9.5 seconds. There are 50 of them per activity. Sixteen longer files chain activities so the transition matrix has real changes to learn. Rates are mixed across 40, 50, and 100 Hz. Every file is interpolated onto a 50 Hz grid before windowing, which is how different phones are brought onto one clock.

The window is 1.0 second (50 samples) and the hop is 0.5 second. A step is about half a second, so the window covers at least one step and the native frequency bin is 1 Hz. The label of a window is the majority activity inside it. Ten files per activity, plus four mixed files, are held out. The scaler and the HMM are fit on the rest.


In [ ]:
splits = load_splits(ROOT)
train_n = sum(1 for item in splits["train"] if item["meta"]["activity"] != "mixed")
test_n = sum(1 for item in splits["test"] if item["meta"]["activity"] != "mixed")
mixed_train = sum(1 for item in splits["train"] if item["meta"]["activity"] == "mixed")
mixed_test = sum(1 for item in splits["test"] if item["meta"]["activity"] == "mixed")
print(f"Single-activity files: {train_n} train, {test_n} test")
print(f"Mixed sequences: {mixed_train} train, {mixed_test} test")
print(f"Window {WINDOW / TARGET_FS:.1f}s, hop {HOP / TARGET_FS:.1f}s, rate {TARGET_FS:.0f} Hz")

examples = {}
for name in ACTIVITIES:
    candidates = [item for item in splits["test"] if item["meta"]["activity"] == name]
    match = max(candidates, key=lambda item: item["duration"])
    mask = match["time"] <= 8.0
    examples[name] = (match["time"][mask], match["magnitude"][mask])
plot_raw_magnitude(examples)


## Features

Gravity in a window is the mean acceleration vector, and it is removed before the magnitude features. Otherwise every activity looks like 1 g and the tilt of the phone dominates the Gaussian.

Five features are time-domain: mean, variance, and RMS of the dynamic magnitude; signal magnitude area; and the correlation of the two horizontal axes. Four are taken from an FFT. The dominant frequency is the peak on the axis that moves the most, after the FFT is zero-padded, and it is reported as 0 Hz when no bin clears the noise floor. Taking that peak on the magnitude would rectify the signal and hide the cadence. Spectral energy stays on the magnitude, from 0.15 to 8 Hz, because that quantity is total motion rather than the motion of one axis. The same peak is computed for the gyroscope.

Z-score mean and scale come from the training windows only. A correlation, a frequency, and a spectral energy do not share a unit.


In [ ]:
raw = []
owners = []
for item in splits["train"]:
    if item["meta"]["activity"] == "mixed":
        continue
    raw.append(item["features"])
    owners.append(np.full(len(item["features"]), item["meta"]["activity"]))
raw = np.concatenate(raw)
owners = np.concatenate(owners)
header = f"{'feature':22} " + " ".join(f"{name:>10}" for name in ACTIVITIES)
print(header)
for col, feature in enumerate(FEATURE_NAMES):
    means = [raw[owners == name, col].mean() for name in ACTIVITIES]
    print(f"{feature:22} " + " ".join(f"{value:10.3f}" for value in means))
print()
print("Walking is the only activity with a stable acceleration cadence near 1.8 Hz.")
print("Jumping has no clean peak (dominant frequency 0) and a much larger spectral energy.")


## Which algorithm the metrics use

Baum-Welch is the learner. The recordings are labeled, and those labels initialize each state, but the deployed phone will not have a label. EM is what updates the start vector, the transitions, and the Gaussians from the windows themselves. The stop test is a relative change in training log-likelihood below 10⁻⁶.

Viterbi is the decoder the table scores. Sensitivity, specificity, and accuracy count one hard label per window, which is the label a live display would show. Forward-backward posteriors are what the E-step uses, and they are the right object when a window should stay uncertain. They are not what the confusion matrix counts. An error in the Viterbi path is an off-diagonal cell.


In [ ]:
model, scaler, history, order, _supervised = fit_model(splits)
rose = all(history[i] <= history[i + 1] + 1e-3 for i in range(len(history) - 1))
rel = abs(history[-1] - history[-2]) / (abs(history[-2]) + 1e-8)
print(f"Iterations: {len(history)}")
print(f"Log-likelihood: {history[0]:.1f} -> {history[-1]:.1f}")
print(f"Monotonic: {rose}. Final relative change: {rel:.3e}")
print("State order after alignment:", [ACTIVITIES[i] for i in order.tolist()])
plot_likelihood(history)


In [ ]:
plot_heatmap(model.A, ACTIVITIES, "Transition matrix", "To", "From", ".2f")
print("Rows sum to", np.round(model.A.sum(axis=1), 6))


## Held-out evaluation

Single-activity test files are new recordings of the same four states. Mixed test files are new chains. Neither was used to fit the scaler or the model, and neither was used to name the states.


In [ ]:
def decode(item):
    return model.viterbi(scaler.transform(item["features"]))

pure = [item for item in splits["test"] if item["meta"]["activity"] != "mixed"]
mixed = [item for item in splits["test"] if item["meta"]["activity"] == "mixed"]

truth = np.concatenate([item["labels"] for item in pure])
predicted = np.concatenate([decode(item) for item in pure])
matrix = confusion_matrix(truth, predicted, len(ACTIVITIES))
metrics = per_class_metrics(matrix)

print(f"{'Activity':12} {'Windows':>8} {'Sensitivity':>12} {'Specificity':>12} {'Accuracy':>10}")
for name, row in zip(ACTIVITIES, metrics):
    print(
        f"{name:12} {row['support']:8d} {row['sensitivity']:12.3f} "
        f"{row['specificity']:12.3f} {row['overall_accuracy']:10.3f}"
    )
file_hits = []
for item in pure:
    path = decode(item)
    file_hits.append(int(np.bincount(path).argmax() == np.bincount(item["labels"]).argmax()))
print(f"Files correct by majority vote: {sum(file_hits)} / {len(file_hits)}")

mixed_truth = np.concatenate([item["labels"] for item in mixed])
mixed_pred = np.concatenate([decode(item) for item in mixed])
mixed_acc = float(np.mean(mixed_truth == mixed_pred))
print(f"Mixed-sequence frame accuracy: {mixed_acc:.3f} on {len(mixed_truth)} windows, {len(mixed)} files")
plot_heatmap(
    matrix,
    ACTIVITIES,
    "Confusion matrix on held-out single-activity files",
    "Predicted",
    "Recorded",
    "d",
)


In [ ]:
example = mixed[0]
path = decode(example)
plot_decoded(example["centers"], example["labels"], path, ACTIVITIES)
print(example["meta"]["path"])


## Reading the result

On the single-activity holdout the path is right on 583 of 585 windows, and every file is right by majority vote. The two errors are standing windows read as still. Walking and jumping are not confused with anything on those files.

The transition matrix is almost diagonal (self-transitions about 0.97 to 0.98). That matches a half-second hop: the state usually stays put. The off-diagonal mass that remains is small. Several scripted changes, including walking straight into still, sit on the numerical floor. One boundary frame against a long run is easy for EM to treat as noise.

The mixed-file plot shows the same limit on a single path. The chain follows the recorded activities, then inserts a short still during standing. A one-second window at a boundary is a mixture, and the transition cost of a brief detour is low.

Standing against still is the pair the features do not finish. Both have almost no spectral energy. Walking is the cadence class, near 1.8 Hz on both the accelerometer and the gyroscope. Jumping is the energy class: no tonal peak, and spectral energy an order of magnitude above walking. Re-run this notebook after the recordings are in `data/`. The rate can drift, the phone can rotate inside the gravity window, and the four face-visible videos this brief requires are still to be recorded.
